<a href="https://colab.research.google.com/github/mekhyal/ML-based-for-detecting-money-laundring-using-SAML-D/blob/main/Preprocessing_SAML_D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P2 – Corrected Preprocessing for SAML-D

This notebook fixes the leakage issue by:

- splitting **before** fitting any preprocessing objects
- removing target-like / leakage-prone columns
- fitting imputation, scaling, and encoding on **train only**
- transforming validation and test sets with the same fitted preprocessor
- saving clean files for direct use in the P3 notebook

**Expected input file:** `SAML-D.csv`


In [ ]:
import kagglehub
path = kagglehub.dataset_download("berkanoztas/synthetic-transaction-monitoring-dataset-aml")


100%|██████████| 193M/193M [00:15<00:00, 13.0MB/s]

Extracting files...


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as statsh
import zipfile
import os
from sklearn.preprocessing import OneHotEncoder
import joblib
from sklearn.preprocessing import StandardScaler
import re
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [ ]:
!pip install -q kaggle
!mkdir -p /content/data
!kaggle datasets download -d berkanoztas/synthetic-transaction-monitoring-dataset-aml -p /content/data

Dataset URL: https://www.kaggle.com/datasets/berkanoztas/synthetic-transaction-monitoring-dataset-aml
License(s): CC-BY-NC-SA-4.0
100% 193M/193M [00:14<00:00, 14.4MB/s]



In [ ]:
!unzip -o /content/data/synthetic-transaction-monitoring-dataset-aml.zip -d /content/data

Archive:  /content/data/synthetic-transaction-monitoring-dataset-aml.zip
  inflating: /content/data/SAML-D.csv  


In [ ]:
import os
df = pd.read_csv(os.path.join(path, 'SAML-D.csv'))

In [ ]:
# # =========================
# # 1) Load the dataset
# # =========================

# possible_paths = [
#     "SAML-D.csv",
#     "/content/SAML-D.csv",
#     "/mnt/data/SAML-D.csv"
# ]

# data_path = None
# for p in possible_paths:
#     if os.path.exists(p):
#         data_path = p
#         break

# if data_path is None:
#     raise FileNotFoundError(
#         "SAML-D.csv was not found. Upload it to Colab or place it in the current working directory."
#     )

# df_raw = pd.read_csv(data_path)
# print("Loaded:", data_path)
# print("Shape:", df_raw.shape)
# display(df_raw.head())

import os
df_raw = pd.read_csv(os.path.join(path, 'SAML-D.csv'))

display(df_raw.head())

# 1.2b: (The Shape): number of rows and columns

print(f"The shape/size:{df_raw.shape}")

fraud = df_raw[df_raw['Is_laundering'] == 1]
not_fraud = df_raw[df_raw['Is_laundering'] == 0]

print("fraud clases: {}".format(len(df_raw[df_raw['Is_laundering'] == 1])))
print("not fraud clases: {}".format(len(df_raw[df_raw['Is_laundering'] == 0])))

,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits
1,10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out
2,10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Small_Fan_Out
3,10:35:21,2022-10-07,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_In
4,10:35:21,2022-10-07,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits


The shape/size:(9504852, 12)
fraud clases: 9873
not fraud clases: 9494979


In [ ]:
print("Amount details of the fraud trans")
not_fraud.Amount.describe()

Amount details of the fraud trans


,Amount
count,9.494979e+06
mean,8.729876e+03
std,2.175003e+04
min,3.730000e+00
25%,2.142930e+03
50%,6.114630e+03
75%,1.045895e+04
max,9.999622e+05


In [ ]:

# =========================
# 2) Basic checks
# =========================

print("Columns:")
print(df_raw.columns.tolist())
print("\nMissing values:")
display(df_raw.isna().sum().sort_values(ascending=False))
print("\nDuplicate rows:", df_raw.duplicated().sum())


Columns:
['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'Is_laundering', 'Laundering_type']

Missing values:


,0
Time,0
Date,0
Sender_account,0
Receiver_account,0
Amount,0
Payment_currency,0
Received_currency,0
Sender_bank_location,0
Receiver_bank_location,0
Payment_type,0



Duplicate rows: 0


## Cleaning helpers

In [ ]:

def clean_value(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip().lower()
    x = re.sub(r'[_\-]+', ' ', x)
    if x in {"na", "n/a", "none", "null", "nan", "missing", ""}:
        return pd.NA
    return x

def normalize_categorical_columns(df):
    df = df.copy()
    cat_cols = [
        'Payment_type',
        'Sender_bank_location',
        'Receiver_bank_location',
        'Laundering_type',
        'Payment_currency',
        'Received_currency'
    ]
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_value)
    return df

def group_currency(x):
    if pd.isna(x):
        return "unknown"
    x = str(x).strip().lower()
    eu_currencies = {'euro', 'swiss franc'}
    middle_east = {'dirham', 'turkish lira', 'moroccan dirham'}
    asia = {'indian rupee', 'pakistani rupee', 'yen'}
    africa = {'naira'}
    americas = {'us dollar', 'mexican peso'}

    if x == 'uk pounds':
        return 'uk_pounds'
    elif x in eu_currencies:
        return 'europe'
    elif x in middle_east:
        return 'middle_east'
    elif x in asia:
        return 'asia'
    elif x in africa:
        return 'africa'
    elif x in americas:
        return 'americas'
    else:
        return 'other_currency'

def group_bank_location(x):
    if pd.isna(x):
        return "unknown"
    x = str(x).strip().lower()
    return "uk" if x == "uk" else "international"

def group_laundering_type(x):
    # This is created only for analysis / sanity checking, NOT for modeling.
    if pd.isna(x):
        return "unknown"

    x = str(x).strip().lower().replace("_", " ").replace("-", " ")

    normal_fan = {
        'normal fan in', 'normal fan out'
    }
    normal_group = {
        'normal single', 'normal group'
    }
    cash_like = {
        'cash withdrawal', 'cash deposit'
    }
    suspicious_structuring = {
        'stack', 'smurfing', 'structuring'
    }

    if x in normal_fan:
        return "normal_fan"
    elif x in normal_group:
        return "normal_group"
    elif x in cash_like:
        return "cash_pattern"
    elif x in suspicious_structuring:
        return "suspicious_structuring"
    else:
        return "other_pattern"


In [ ]:

# =========================
# 3) Lightweight cleaning + feature engineering on raw data
#    (still BEFORE model preprocessing, but this part does NOT fit on the data)
# =========================

df = df_raw.copy()
df = normalize_categorical_columns(df)

# Target cleanup
df['Is_laundering'] = df['Is_laundering'].fillna(0).astype(int)

# Numeric cleanup
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df['Amount'] = df['Amount'].fillna(df['Amount'].median())

# Feature engineering
df['Amount_log'] = np.log1p(df['Amount'])

# safer quartile feature; convert to string category later
df['Amount_Q4'] = pd.qcut(df['Amount'], q=4, duplicates='drop')
df['Amount_Q4'] = df['Amount_Q4'].astype(str)

# grouped / reduced-cardinality features
df['Sender_bank_location_grouped'] = df['Sender_bank_location'].apply(group_bank_location)
df['Receiver_bank_location_grouped'] = df['Receiver_bank_location'].apply(group_bank_location)
df['Payment_currency_grouped'] = df['Payment_currency'].apply(group_currency)
df['Received_currency_grouped'] = df['Received_currency'].apply(group_currency)

# analysis-only helper column
if 'Laundering_type' in df.columns:
    df['Laundering_type_grouped'] = df['Laundering_type'].apply(group_laundering_type)

print("Prepared dataframe shape:", df.shape)
display(df.head())


Prepared dataframe shape: (9504852, 19)


,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type,Amount_log,Amount_Q4,Sender_bank_location_grouped,Receiver_bank_location_grouped,Payment_currency_grouped,Received_currency_grouped,Laundering_type_grouped
0,10:35:19,2022-10-07,8724731955,2769355426,1459.15,uk pounds,uk pounds,uk,uk,cash deposit,0,normal cash deposits,7.286294,"(3.729, 2143.688]",uk,uk,uk_pounds,uk_pounds,other_pattern
1,10:35:20,2022-10-07,1491989064,8401255335,6019.64,uk pounds,dirham,uk,uae,cross border,0,normal fan out,8.702949,"(2143.688, 6113.72]",uk,international,uk_pounds,middle_east,normal_fan
2,10:35:20,2022-10-07,287305149,4404767002,14328.44,uk pounds,uk pounds,uk,uk,cheque,0,normal small fan out,9.570071,"(10458.462, 12618498.4]",uk,uk,uk_pounds,uk_pounds,other_pattern
3,10:35:21,2022-10-07,5376652437,9600420220,11895.00,uk pounds,uk pounds,uk,uk,ach,0,normal fan in,9.383957,"(10458.462, 12618498.4]",uk,uk,uk_pounds,uk_pounds,normal_fan
4,10:35:21,2022-10-07,9614186178,3803336972,115.25,uk pounds,uk pounds,uk,uk,cash deposit,0,normal cash deposits,4.755743,"(3.729, 2143.688]",uk,uk,uk_pounds,uk_pounds,other_pattern


## Define leakage-prone columns to remove

These are excluded from modeling because they are IDs, direct label-like descriptors, or redundant raw columns after feature engineering.


In [ ]:

# =========================
# 4) Drop leakage-prone columns and define X / y
# =========================

target_col = 'Is_laundering'

drop_for_model = [
    # direct identifiers
    'Sender_account',
    'Receiver_account',

    # raw columns replaced with grouped features
    'Sender_bank_location',
    'Receiver_bank_location',
    'Payment_currency',
    'Received_currency',

    # strong leakage / target-like columns
    'Laundering_type',
    'Laundering_type_grouped',

    # raw fields not used after feature engineering
    'Amount',
    'Date',
    'Time'
]

drop_for_model = [c for c in drop_for_model if c in df.columns]

X = df.drop(columns=[target_col] + drop_for_model, errors='ignore')
y = df[target_col].copy()

print("Target distribution:")
print(y.value_counts(dropna=False))
print("\nModel feature columns:")
print(X.columns.tolist())
print("\nNumber of model features before encoding:", X.shape[1])


Target distribution:
Is_laundering
0    9494979
1       9873
Name: count, dtype: int64

Model feature columns:
['Payment_type', 'Amount_log', 'Amount_Q4', 'Sender_bank_location_grouped', 'Receiver_bank_location_grouped', 'Payment_currency_grouped', 'Received_currency_grouped']

Number of model features before encoding: 7


In [ ]:

# Extra sanity check for suspicious columns that should NOT go into the model
suspicious_cols = [c for c in X.columns if any(k in c.lower() for k in ['launder', 'fraud', 'susp'])]
print("Suspicious columns still in X:", suspicious_cols)


Suspicious columns still in X: []


In [ ]:

# =========================
# 5) Train / validation / test split (70 / 15 / 15)
# =========================

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=42
)

val_size = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=val_size,
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts())
print("\nVal class distribution:")
print(y_val.value_counts())
print("\nTest class distribution:")
print(y_test.value_counts())


Train: (6653396, 7) (6653396,)
Val:   (1425728, 7) (1425728,)
Test:  (1425728, 7) (1425728,)

Train class distribution:
Is_laundering
0    6646485
1       6911
Name: count, dtype: int64

Val class distribution:
Is_laundering
0    1424247
1       1481
Name: count, dtype: int64

Test class distribution:
Is_laundering
0    1424247
1       1481
Name: count, dtype: int64


## Fit preprocessing on train only

In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

# =========================
# 6) Build preprocessing pipeline
# =========================

categorical_features = [c for c in [
    'Payment_type',
    'Sender_bank_location_grouped',
    'Receiver_bank_location_grouped',
    'Payment_currency_grouped',
    'Received_currency_grouped',
    'Amount_Q4'
] if c in X_train.columns]

numeric_features = [c for c in [
    'Amount_log'
] if c in X_train.columns]

# keep any additional numeric columns if present
for col in X_train.columns:
    if col not in categorical_features and pd.api.types.is_numeric_dtype(X_train[col]) and col not in numeric_features:
        numeric_features.append(col)

# keep any additional leftover object columns as categorical
for col in X_train.columns:
    if col not in categorical_features and col not in numeric_features:
        categorical_features.append(col)

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# Fit ONLY on training data
preprocessor.fit(X_train)

print("Preprocessor fitted on training data only.")


Categorical features: ['Payment_type', 'Sender_bank_location_grouped', 'Receiver_bank_location_grouped', 'Payment_currency_grouped', 'Received_currency_grouped', 'Amount_Q4']
Numeric features: ['Amount_log']
Preprocessor fitted on training data only.


In [ ]:
# =========================
# 7) Transform train / val / test
# =========================
from scipy import sparse

X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Processed train shape:", X_train_processed.shape)
print("Processed val shape:  ", X_val_processed.shape)
print("Processed test shape: ", X_test_processed.shape)

print("Train sparse:", sparse.issparse(X_train_processed))
print("Val sparse:  ", sparse.issparse(X_val_processed))
print("Test sparse: ", sparse.issparse(X_test_processed))

# show only a tiny sample safely
sample_dense = pd.DataFrame(
    X_train_processed[:5].toarray(),
    columns=feature_names
)
display(sample_dense.head())

Processed train shape: (6653396, 30)
Processed val shape:   (1425728, 30)
Processed test shape:  (1425728, 30)
Train sparse: True
Val sparse:   True
Test sparse:  True


,num__Amount_log,cat__Payment_type_ach,cat__Payment_type_cash deposit,cat__Payment_type_cash withdrawal,cat__Payment_type_cheque,cat__Payment_type_credit card,cat__Payment_type_cross border,cat__Payment_type_debit card,cat__Sender_bank_location_grouped_international,cat__Sender_bank_location_grouped_uk,...,cat__Received_currency_grouped_americas,cat__Received_currency_grouped_asia,cat__Received_currency_grouped_europe,cat__Received_currency_grouped_middle_east,cat__Received_currency_grouped_other_currency,cat__Received_currency_grouped_uk_pounds,"cat__Amount_Q4_(10458.462, 12618498.4]","cat__Amount_Q4_(2143.688, 6113.72]","cat__Amount_Q4_(3.729, 2143.688]","cat__Amount_Q4_(6113.72, 10458.462]"
0,-1.063112,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,-1.846661,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.032158,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,0.195514,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,0.128033,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [ ]:
# =========================
# 8) Leakage / overlap sanity checks
# =========================

print("Any suspicious processed columns?")
print([c for c in feature_names if any(k in c.lower() for k in ['launder', 'fraud', 'susp'])])

# lightweight overlap check on a small sample of ORIGINAL rows only
train_sample = X_train.head(1000).astype(str)
test_sample = X_test.head(1000).astype(str)

train_rows = set(map(tuple, train_sample.to_numpy()))
test_rows = set(map(tuple, test_sample.to_numpy()))
overlap = len(train_rows.intersection(test_rows))

print("Sample overlap between original train and test rows:", overlap)

Any suspicious processed columns?
[]
Sample overlap between original train and test rows: 0


In [ ]:
# =========================
# 9) Save outputs for P3
# =========================

import os
import json
import joblib
from scipy import sparse

output_dir = "p2_outputs"
os.makedirs(output_dir, exist_ok=True)

# save sparse matrices
sparse.save_npz(os.path.join(output_dir, "X_train_processed.npz"), X_train_processed)
sparse.save_npz(os.path.join(output_dir, "X_val_processed.npz"), X_val_processed)
sparse.save_npz(os.path.join(output_dir, "X_test_processed.npz"), X_test_processed)

# save targets
y_train.to_csv(os.path.join(output_dir, "y_train.csv"), index=False)
y_val.to_csv(os.path.join(output_dir, "y_val.csv"), index=False)
y_test.to_csv(os.path.join(output_dir, "y_test.csv"), index=False)

# save preprocessor
joblib.dump(preprocessor, os.path.join(output_dir, "preprocessor.pkl"))

# save feature names
with open(os.path.join(output_dir, "feature_names.json"), "w") as f:
    json.dump(list(feature_names), f)

metadata = {
    "target_column": target_col,
    "raw_rows": int(df.shape[0]),
    "raw_columns_after_feature_engineering": int(df.shape[1]),
    "model_input_columns_before_encoding": list(X.columns),
    "processed_feature_count": int(len(feature_names)),
    "train_shape": list(X_train_processed.shape),
    "val_shape": list(X_val_processed.shape),
    "test_shape": list(X_test_processed.shape),
    "dropped_for_model": drop_for_model
}

with open(os.path.join(output_dir, "preprocessing_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved files in:", output_dir)
print(sorted(os.listdir(output_dir)))

Saved files in: p2_outputs
['X_test_processed.npz', 'X_train_processed.npz', 'X_val_processed.npz', 'feature_names.json', 'preprocessing_metadata.json', 'preprocessor.pkl', 'y_test.csv', 'y_train.csv', 'y_val.csv']


## Loader cell for the P3 notebook

Copy this cell into the P3 notebook if needed.


In [ ]:
# Example loader for P3

import os
import json
import joblib
import pandas as pd
from scipy import sparse

output_dir = "p2_outputs"

X_train = sparse.load_npz(os.path.join(output_dir, "X_train_processed.npz"))
X_val = sparse.load_npz(os.path.join(output_dir, "X_val_processed.npz"))
X_test = sparse.load_npz(os.path.join(output_dir, "X_test_processed.npz"))

y_train = pd.read_csv(os.path.join(output_dir, "y_train.csv")).squeeze("columns")
y_val = pd.read_csv(os.path.join(output_dir, "y_val.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(output_dir, "y_test.csv")).squeeze("columns")

with open(os.path.join(output_dir, "feature_names.json"), "r") as f:
    feature_names = json.load(f)

preprocessor = joblib.load(os.path.join(output_dir, "preprocessor.pkl"))

print("Loaded processed sets:")
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)
print("Number of features:", len(feature_names))

Loaded processed sets:
Train: (6653396, 30) (6653396,)
Val:   (1425728, 30) (1425728,)
Test:  (1425728, 30) (1425728,)
Number of features: 30
